In [2]:
!pip install pandas datasets

  Using cached pandas-3.0.6-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached pyarrow-25.0.1-cp311-cp311-macosx_12_0_arm64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-4.0.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (17 kB)
  Using cached multiprocess-0.70.19-py311-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached aiohttp-3.14.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (8.3

In [3]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("Riksarkivet/swener_1800")
ds

/Users/aau/Desktop/swener_1800/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating eval split: 100%|██████████| 170/170 [00:00<00:00, 38840.38 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'entity_types', 'entity_start_chars', 'entity_end_chars', 'id', 'word_start_chars', 'word_end_chars'],
        num_rows: 677
    })
    eval: Dataset({
        features: ['text', 'entity_types', 'entity_start_chars', 'entity_end_chars', 'id', 'word_start_chars', 'word_end_chars'],
        num_rows: 170
    })
})

In [5]:
for name, d in ds.items():
    lens = pd.Series([len(t) for t in d["text"]])
    ents = pd.Series([len(e) for e in d["entity_types"]])
    print(name, "-", len(d), "docs")
    print("  chars/doc:   ", lens.describe()[["min", "mean", "max"]].round(0).to_dict())
    print("  entities/doc:", ents.describe()[["min", "mean", "max"]].round(1).to_dict(), "| total:", ents.sum())

train - 677 docs
  chars/doc:    {'min': 8.0, 'mean': 3653.0, 'max': 140295.0}
  entities/doc: {'min': 0.0, 'mean': 38.9, 'max': 687.0} | total: 26341
eval - 170 docs
  chars/doc:    {'min': 118.0, 'mean': 3075.0, 'max': 36805.0}
  entities/doc: {'min': 3.0, 'mean': 42.4, 'max': 594.0} | total: 7215


In [6]:
from collections import Counter

counts = {name: Counter(t for types in d["entity_types"] for t in types) for name, d in ds.items()}
df = pd.DataFrame(counts).fillna(0).astype(int)
df["train_%"] = (df["train"] / df["train"].sum() * 100).round(1)
df["eval_%"] = (df["eval"] / df["eval"].sum() * 100).round(1)
df.sort_values("train", ascending=False)

,train,eval,train_%,eval_%
PER,9387,2488,35.6,34.5
LOC,4552,1356,17.3,18.8
OCC,3708,1065,14.1,14.8
TME-DATE,2768,777,10.5,10.8
MSR-MON,1476,471,5.6,6.5
ORG-INST,1390,365,5.3,5.1
TME-TIME,961,215,3.6,3.0
WRK,780,160,3.0,2.2
EVN,290,60,1.1,0.8
ORG-COMP,255,69,1.0,1.0
